# Storm motion baseline

This notebook is the first end-to-end exercise for the Stormageddon storm-motion-tracking problem. It creates two radar-like reflectivity scans, identifies storm cells, estimates their motion, and extrapolates a short forecast.

It intentionally uses synthetic frames first: that makes the tracking logic inspectable before we introduce real NEXRAD Level II data and its data-quality complications.

## What this baseline does — and does not do

It thresholds reflectivity, labels connected regions, and matches each cell to the closest eligible cell in the next scan. It then uses a constant-velocity forecast. This is a baseline, not an operational storm-warning model: it does **not** yet represent cell growth, splits, merges, or forecast uncertainty.

In [ ]:
# Run this once after opening the notebook. It installs the local project plus
# the optional Py-ART reader used later for real NEXRAD Level II files.
%pip install -e "../.[nexrad]"

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

# Make the repo's src/ package importable even before the kernel is restarted.
repo = Path.cwd().parent
sys.path.insert(0, str(repo / 'src'))

from storm_motion.tracking import RadarFrame, detect_cells, match_cells

print(f'Using repository: {repo}')

In [ ]:
# Build two simple, radar-like 100 km x 100 km reflectivity fields.
# The cell moves 8 km east and 4 km north in ten minutes.
axis_m = np.linspace(-50_000, 50_000, 300)
x_m, y_m = np.meshgrid(axis_m, axis_m)

def gaussian_cell(center_x_m, center_y_m, peak_dbz=55.0, radius_m=7_000.0):
    distance_squared = (x_m - center_x_m) ** 2 + (y_m - center_y_m) ** 2
    return peak_dbz * np.exp(-distance_squared / (2 * radius_m ** 2))

observed_at = datetime(2026, 5, 1, 20, 0, tzinfo=timezone.utc)
previous_frame = RadarFrame(
    observed_at=observed_at,
    reflectivity_dbz=gaussian_cell(-15_000, -10_000),
    x_m=x_m,
    y_m=y_m,
)
current_frame = RadarFrame(
    observed_at=observed_at + timedelta(minutes=10),
    reflectivity_dbz=gaussian_cell(-7_000, -6_000),
    x_m=x_m,
    y_m=y_m,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
for ax, frame, title in zip(axes, [previous_frame, current_frame], ['Scan t0', 'Scan t+10 min']):
    image = ax.pcolormesh(frame.x_m / 1_000, frame.y_m / 1_000, frame.reflectivity_dbz, cmap='turbo', vmin=0, vmax=60)
    ax.set(title=title, xlabel='km east', ylabel='km north', aspect='equal')
fig.colorbar(image, ax=axes, label='reflectivity (dBZ)')
plt.show()

In [ ]:
threshold_dbz = 35.0
previous_cells = detect_cells(previous_frame, threshold_dbz=threshold_dbz)
current_cells = detect_cells(current_frame, threshold_dbz=threshold_dbz)
matches = match_cells(previous_cells, current_cells, elapsed_seconds=10 * 60)

print('previous cells:', previous_cells)
print('current cells:', current_cells)
print('motion matches:', matches)

In [ ]:
# Visualize the estimated motion vector on the newest scan.
match = matches[0]
old_cell, new_cell = previous_cells[0], current_cells[0]
dx_m = new_cell.centroid_x_m - old_cell.centroid_x_m
dy_m = new_cell.centroid_y_m - old_cell.centroid_y_m

plt.figure(figsize=(7, 6))
plt.pcolormesh(current_frame.x_m / 1_000, current_frame.y_m / 1_000, current_frame.reflectivity_dbz, cmap='turbo', vmin=0, vmax=60)
plt.arrow(old_cell.centroid_x_m / 1_000, old_cell.centroid_y_m / 1_000, dx_m / 1_000, dy_m / 1_000, width=0.35, color='white', length_includes_head=True)
plt.scatter([old_cell.centroid_x_m / 1_000, new_cell.centroid_x_m / 1_000], [old_cell.centroid_y_m / 1_000, new_cell.centroid_y_m / 1_000], c=['white', 'black'], s=70)
plt.colorbar(label='reflectivity (dBZ)')
plt.title(f'Estimated speed: {match.speed_mps:.1f} m/s; bearing: {match.bearing_degrees:.0f}°')
plt.xlabel('km east'); plt.ylabel('km north'); plt.axis('equal'); plt.show()

In [ ]:
# Map-style, constant-velocity nowcast for the next 10, 20, and 30 minutes.
forecast_minutes = np.array([10, 20, 30])
forecast_x_m = new_cell.centroid_x_m + (dx_m / 600) * forecast_minutes * 60
forecast_y_m = new_cell.centroid_y_m + (dy_m / 600) * forecast_minutes * 60

fig, ax = plt.subplots(figsize=(8, 7))
image = ax.pcolormesh(current_frame.x_m / 1_000, current_frame.y_m / 1_000, current_frame.reflectivity_dbz, cmap='turbo', vmin=0, vmax=60)
ax.plot(
    [new_cell.centroid_x_m / 1_000, *list(forecast_x_m / 1_000)],
    [new_cell.centroid_y_m / 1_000, *list(forecast_y_m / 1_000)],
    '--', color='white', linewidth=2, label='projected cell path',
)
ax.scatter(new_cell.centroid_x_m / 1_000, new_cell.centroid_y_m / 1_000, s=115, c='white', edgecolors='black', zorder=3, label='current cell center')
ax.scatter(forecast_x_m / 1_000, forecast_y_m / 1_000, s=100, c='#ffdf00', edgecolors='black', zorder=3, label='forecast center')
for minutes, x, y in zip(forecast_minutes, forecast_x_m, forecast_y_m):
    ax.annotate(f'+{minutes} min', (x / 1_000, y / 1_000), xytext=(7, 7), textcoords='offset points', color='white', weight='bold')
ax.set(title='Storm-cell nowcast: constant-velocity projection', xlabel='km east', ylabel='km north', aspect='equal')
ax.legend(loc='upper left')
fig.colorbar(image, ax=ax, label='reflectivity (dBZ)')
plt.show()

# Next milestone: replace the synthetic frames with two files loaded using
# storm_motion.nexrad.load_lowest_reflectivity_sweep, then quantify error
# against later scans and add split/merge plus uncertainty handling.